## Sub-Phase 3.1 & Phase 5 Setup

In [ ]:
# 0. Mount Google Drive for Persistent Storage
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# Create dedicated directories in your Drive
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Hiligaynon_NER_Project'
DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, 'logs'), exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Google Drive mounted. Project data will be saved to: {DRIVE_PROJECT_DIR}")
print(f"ACTION REQUIRED: Ensure your .conll files are uploaded to: {DATA_DIR}")

In [ ]:
# 1. Install Required Libraries
!pip install -q transformers datasets seqeval evaluate spacy sklearn-crfsuite
print("Dependencies installed.")

import numpy as np
import evaluate
from datasets import Dataset, DatasetDict

In [ ]:
# 2. Model & Tokenizer Initialization
# (Ensure model_xlmr.py is uploaded to your Colab directory)
try:
    from model_xlmr import initialize_xlmr_model
except ModuleNotFoundError:
    print("WARNING: 'model_xlmr.py' not found! Make sure to upload it to the Colab runtime.")

# UPDATED: Full 29-label BIOES permutations for the 7 new target classes
label_list = [
    "O",
    "B-PERSON",   "I-PERSON",   "E-PERSON",   "S-PERSON",
    "B-ORG",      "I-ORG",      "E-ORG",      "S-ORG",
    "B-LOCATION", "I-LOCATION", "E-LOCATION", "S-LOCATION",
    "B-DATETIME", "I-DATETIME", "E-DATETIME", "S-DATETIME",
    "B-MONEY",    "I-MONEY",    "E-MONEY",    "S-MONEY",
    "B-EVENT",    "I-EVENT",    "E-EVENT",    "S-EVENT",
    "B-NORP",     "I-NORP",     "E-NORP",     "S-NORP"
]

print("Initializing Model...")
model, tokenizer, config = initialize_xlmr_model(label_list)

In [ ]:
# 3. Data Extraction (Parsing CoNLL to Hugging Face Datasets)
def read_conll(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"CRITICAL ERROR: Cannot find {file_path}. Please upload it to your Google Drive data folder.")

    with open(file_path, "r", encoding="utf-8") as f:
        sentences, labels, current_words, current_labels = [], [], [], []
        for line in f:
            if line.startswith("-DOCSTART-") or line == "" or line == "\n":
                if current_words:
                    sentences.append(current_words)
                    labels.append(current_labels)
                    current_words, current_labels = [], []
            else:
                # UPDATED: Dynamic split to handle both tabs and spaces safely
                splits = line.strip().split()
                if len(splits) >= 2:
                    current_words.append(splits[0])
                    current_labels.append(splits[1])
        if current_words:
            sentences.append(current_words)
            labels.append(current_labels)
    return {"tokens": sentences, "ner_tags": labels}

# Explicitly target the new 3-file structure
TRAIN_PATH = os.path.join(DATA_DIR, "dataset_train_final.conll")
VAL_PATH = os.path.join(DATA_DIR, "dataset_val_final.conll")
TEST_PATH = os.path.join(DATA_DIR, "dataset_test_final.conll")

print("Loading datasets...")
train_data = read_conll(TRAIN_PATH)
val_data = read_conll(VAL_PATH)
test_data = read_conll(TEST_PATH)

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(val_data),
    "test": Dataset.from_dict(test_data)     # <--- Test set is now officially registered
})

print(f"Loaded {len(dataset['train'])} train, {len(dataset['validation'])} validation, and {len(dataset['test'])} test samples.")

## Sub-Phase 4.1: Environment Setup & Hyperparameter Tuning

In [ ]:
# 4. Input Tokenization and Alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # -100 tells PyTorch to ignore this token during loss calculation
            elif word_idx != previous_word_idx:
                # UPDATED: Strict tag checking to catch typos in the dataset early
                current_tag = label[word_idx]
                if current_tag not in config.label2id:
                    print(f"WARNING: Unrecognized tag '{current_tag}' found. Defaulting to 'O'.")
                label_ids.append(config.label2id.get(current_tag, 0))
            else:
                label_ids.append(-100) # Only label the first subword token
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map the tokenizer across all 3 splits simultaneously
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

train_dataset = tokenized_datasets.get("train")
eval_dataset = tokenized_datasets.get("validation")  # <--- Trainer will use this for epoch evaluation
test_dataset = tokenized_datasets.get("test")        # <--- Held back for final evaluation

## Sub-Phase 4.2: Execution & Checkpointing

In [ ]:
# 5. Entity-Level Evaluation Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [config.id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [config.id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Pipeline Setup Complete! Ready to configure training.")

In [ ]:
import torch
from transformers import TrainingArguments, Trainer

# Set Hyperparameters based on constrained resources (Colab Pro T4/A100)
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# UPDATED: Increased to 8 epochs to handle the complex 29-label boundary distributions
EPOCHS = 8
WARMUP_RATIO = 0.1

training_args = TrainingArguments(
    output_dir=os.path.join(DRIVE_PROJECT_DIR, 'checkpoints'),
    logging_dir=os.path.join(DRIVE_PROJECT_DIR, 'logs'),
    logging_strategy='steps',
    logging_steps=50,
    eval_strategy='epoch', # Fixed deprecation warning here
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    lr_scheduler_type='linear',
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
)

print("TrainingArguments successfully initialized.")

In [ ]:
from transformers import DataCollatorForTokenClassification

def execute_training(model, tokenizer, train_dataset, eval_dataset, compute_metrics):
    data_collator = DataCollatorForTokenClassification(tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    print("Initiating training sequence...")
    print(f"Checkpoints will be saved to: {training_args.output_dir}")

    # Execute the core training loop
    trainer.train()

    final_save_path = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints', 'best_model')
    trainer.save_model(final_save_path)

    print(f"Training finalized. Top-performing model safely preserved at {final_save_path}")
    return trainer

# EXECUTION TRIGGER (Uncommented and active)
trainer = execute_training(model, tokenizer, train_dataset, eval_dataset, compute_metrics)